In [1]:
import os
import random
import numpy as np
import pandas as pd
import plotly.express as px

import torch
import torchaudio
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset

In [2]:
class Spectrogram(nn.Module):
    def __init__(self, sr=32000, n_fft=2048, n_mels=256, hop_length=512, f_min=20, f_max=16000, channels=1, norm="slaney", mel_scale="htk", target_size=(256, 256), top_db=80.0, delta_win=5,):
        super().__init__()
        self.channels = channels
        self.top_db = top_db

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=f_min,
            f_max=f_max,
            mel_scale=mel_scale,
            pad_mode="reflect",
            power=2.0,
            norm=norm,
            center=True,
        )

        print('channels', self.channels)
        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, S):
        amin = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(S))  # fixed ref = 1.0 effectively
        if self.top_db is not None:
            # per-sample top_db clipping: keep dims for broadcasting
            max_val = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x, resize=True):
        squeeze = False
        if x.dim() == 1:
            x = x.unsqueeze(0)
            squeeze = True

        mel_spec = self.mel_transform(x)          # (B, n_mels, time)
        mel_spec = self.power_to_db(mel_spec)

        mel_spec = mel_spec.unsqueeze(1).repeat(1, self.channels, 1, 1)

        if resize: mel_spec = self.resize(mel_spec)           # (B, C, H, W)

        
        B, C = mel_spec.shape[:2]
        flat = mel_spec.view(B, C, -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel_spec = (mel_spec - mins) / (maxs - mins + 1e-7)

        if squeeze:
            mel_spec = mel_spec.squeeze(0)

        return mel_spec

In [3]:
import timm

class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = {
            'scale':1,
            'backbone_pooling':'avg',
            'backbone':'tf_efficientnetv2_b0',
            'dropout':0.1,
            'pretrained':True,
            'channels':1,
            'num_labels':234,
        }
        if config: self.config.update(config)

        self.training = True

        self.backbone = timm.create_model(
            self.config['backbone'], 
            pretrained=self.config['pretrained'],  
            num_classes=self.config['num_labels'],  
            global_pool=self.config['backbone_pooling'],
            in_chans=1,
            drop_rate=self.config['dropout'],
        )
        feature_dim = self.backbone.num_features
        
    def forward(self, x):
        labels = self.backbone(x)
        return labels

In [4]:
class BirdDataset(Dataset):
    PATH = '/kaggle/input/competitions/birdclef-2026/'
    TEST_PATH = PATH + 'test_soundscapes/'
    TRAIN_PATH = PATH + 'train_soundscapes/'
    taxonomy = pd.read_csv(PATH+'taxonomy.csv')
    
    LABELS = list(np.unique(taxonomy.primary_label))
    CLASSES = list(np.unique(taxonomy.class_name))
    BATCH_SIZE = 32

    DUR = 5
    SR = 32000

    def __init__(self, split_size=0.2, seed=2, n_repeat=1, is_train=True):
        paths = [self.TEST_PATH+x for x in os.listdir(self.TEST_PATH) if '.ogg' in x]
        if len(paths)==0:
            paths = [self.TRAIN_PATH+x for x in os.listdir(self.TRAIN_PATH) if '.ogg' in x]
            paths = sorted(paths)[:16]
        df = pd.DataFrame([], index=range(len(paths)*int(60/self.DUR)), columns=['path', 'start', 'end'])
        self.paths = paths.copy()


    def __len__(self):
        return len(self.paths)

In [5]:
dataset = BirdDataset()

In [6]:
#exp_id = '8658fa9ee3393ad66210d15d4fd2063c41d50697e9fe201dbe9153e375ae2abf'

#history = pd.read_csv(f"/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/history/{exp_id}.csv")

#model_path = f"/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models/{exp_id}.pth"

#config = {x: history.iloc[0][x] for x in history.columns[7:] }

In [7]:
exp_id = '8fbf21d88526d274653ab25a636ac8e2f0db31996ebd8ccc1376dd94f3906716 (1)'

history = pd.read_csv(f"/kaggle/input/datasets/unsseo/experiment-model/{exp_id}.csv")

model_path = f"/kaggle/input/datasets/unsseo/experiment-model/{exp_id}.pth"

config = {x: history.iloc[0][x] for x in history.columns[7:] }

In [8]:
def format_time(t):
    h = int(t/3600)
    t -= h*3600
    m = int(t/60)
    t -= m*60

    out = ''
    if h>0: out+=str(h)+':'
    return out+str(m).zfill(2)+':'+str(int(t)).zfill(2)


def decode_config(cfg):
    for X in cfg:
        try: cfg[X] = eval(cfg[X])
        except:None
    return cfg


def predict(filepath):
    wav, sr = torchaudio.load(filepath)
    n_seg = int(60/dataset.DUR)
    wav = wav.float()[:,:dataset.SR*60]
    n_repeat = 1
    wav = wav.float().reshape((n_seg, dataset.SR*dataset.DUR))
    
    activation = nn.Sigmoid()
    PREDS = []
    with torch.no_grad():
        mel = torch.stack([spec(wav[i]) for i in range(len(wav))])
        for _ in range(n_repeat):
            PREDS.append(activation(model(mel).unsqueeze(0)))
    PREDS = torch.concat(PREDS)
    pred = torch.mean(PREDS, dim=0)

    names = [ID+'_'+t for ID, t in zip([filepath.split('/')[-1].split('.')[0]]*n_seg, (np.array(range(n_seg))*dataset.DUR+dataset.DUR).astype(str))]

    pred = pred.numpy()
    return pred, names

In [9]:
# The model was trained on the files from "train_audio" folder only
LABELS_audio = np.unique(pd.read_csv(dataset.PATH+'train.csv').primary_label)

In [10]:
from concurrent.futures import ThreadPoolExecutor
from time import time

cfg = decode_config(config)
cfg.update({'pretrained':False})
spec = Spectrogram(**cfg['mel'])
model = BirdModel(config=cfg)
model.load_state_dict(torch.load(model_path, weights_only=True, map_location=torch.device('cpu')))

pred = []
names = []
model.eval()
start = time()
with ThreadPoolExecutor(max_workers=4) as executor:
    for p, n in executor.map(predict, dataset.paths):
        pred.append(p)
        names.append(n)

        fps = len(pred)/(time()-start)
        if len(pred)%16==0: print(np.round(100*len(pred)/len(dataset)), '%', format_time(time()-start), '  -  remaining:', format_time((len(dataset)-len(pred))/fps))
pred = np.concatenate(pred, axis=0)

channels 1
100.0 % 00:05   -  remaining: 00:00


In [11]:
#Pred = pd.DataFrame(np.zeros((len(pred), 234)), columns=dataset.LABELS)
#Pred[LABELS_audio] = pred
#Pred.insert(0, 'row_id', np.concatenate(names, axis=0))

Pred = pd.DataFrame(pred, columns=dataset.LABELS)
Pred.insert(0, 'row_id', np.concatenate(names, axis=0))
Pred.to_csv('submission.csv', index=False)


In [12]:
Pred.to_csv('submission.csv', index=False)